### Installation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    %pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    %pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth
%pip install transformers==4.55.4
%pip install --no-deps trl==0.22.2

# Imports

In [ ]:
from unsloth import FastLanguageModel
import torch
import pandas as pd
from datasets import Dataset
import numpy as np
from pathlib import Path
import json
import pyarrow as pa
import pyarrow.parquet as pq

### Unsloth

In [ ]:
max_seq_length = 2048
dtype = None 
load_in_4bit = True

fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

<a name="Data"></a>
### Data Prep

In [ ]:
DATA_PATH = "trn.json"
OUT_DIR   = Path("trn_parquet")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = 100_000
cols = ["title", "content"]

shard_idx = 0
for chunk in pd.read_json(DATA_PATH, lines=True, chunksize=CHUNKSIZE):

    chunk = chunk.dropna(subset=cols).drop_duplicates(subset=cols)
    if chunk.empty:
        continue

    table = pa.Table.from_pandas(chunk[cols], preserve_index=False)
    pq.write_table(table, OUT_DIR / f"part-{shard_idx:05d}.parquet")
    shard_idx += 1

print(f"Wrote {shard_idx} shards to {OUT_DIR}")

In [ ]:
from datasets import load_dataset

parquet_files = str(OUT_DIR / "part-*.parquet")
ds = load_dataset("parquet", data_files=parquet_files, split="train")
ds

In [ ]:
alpaca_prompt = """
### Whats the product description?
Title: {}
Content: {}

### Description:
{}
"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    title = examples["title"]
    content = examples["content"]

    texts = []
    for title, content in zip(title, content):
        text = alpaca_prompt.format(title, content, "") + EOS_TOKEN
        texts.append(text)

    return { "text" : texts, }

In [ ]:
# final_df = df[['title', 'content']].copy()

# final_df['title'] = final_df['title'].str.lower().str.strip()
# final_df['content'] = final_df['content'].str.lower().str.strip()

# final_df.replace('', np.nan, inplace=True)

# final_df.dropna(inplace=True)
# final_df.drop_duplicates(inplace=True)

# print(f"Original DataFrame shape: {df.shape}")
# print(f"Cleaned DataFrame shape:  {final_df.shape}")

In [ ]:
# dataset = Dataset.from_pandas(final_df.reset_index(drop=True), preserve_index=False)
# dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
dataset = ds.map(formatting_prompts_func, batched = True)

<a name="Train"></a>
### Train the model

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1,
        max_steps = 500,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!



In [ ]:
model.save_pretrained("models/lora_model")
tokenizer.save_pretrained("models/lora_model")

In [ ]:
alpaca_prompt_fixed = """### Instruction:
Generate a detailed and informative product description based on the title.

### Input:
{}

### Response:
{}"""

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt_fixed.format(
        "phone bluetooth",
        "",
    )
], return_tensors = "pt").to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if True:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "models/lora_model",
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt_fixed.format(
        "mac mini m4",
        "",
    )
], return_tensors = "pt").to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)